# Chess piece detection — YOLO11n

Colab notebook that trains **Ultralytics YOLO11n** on the Roboflow Universe dataset
[chess-piece-detection-cpgnx / 1](https://universe.roboflow.com/chess-ai-0uukd/chess-piece-detection-cpgnx/dataset/1).

This is a **standalone object-detection experiment**. It does **not** replace Takes's
on-device `PieceClassifier.mlmodel` (that model classifies 64 warped square crops, including `empty`).
Do not drop the exported Core ML detector into the iOS app as a crop classifier.

## How to run

1. Upload this file to [Google Colab](https://colab.research.google.com/).
2. Runtime → Change runtime type → **GPU** (T4 is enough for YOLO11n).
3. Run all cells. You will be prompted for a Roboflow API key (free Universe key is enough).
4. For a long 300-epoch run, set `MOUNT_DRIVE = True` in the next cell so checkpoints survive disconnects.

Trained artifacts: `best.pt`, `last.pt`, Ultralytics plots, and a Core ML `.mlpackage` (if export succeeds on Colab Linux).

## 0. Optional Google Drive mount

In [ ]:
from pathlib import Path
import sys

# Set True for full 300-epoch runs so last.pt / best.pt survive a Colab disconnect.
MOUNT_DRIVE = False

IN_COLAB = "google.colab" in sys.modules

if MOUNT_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive mount is Colab-only. Set MOUNT_DRIVE = False.")
    from google.colab import drive

    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/chess-piece-detection")
elif IN_COLAB:
    WORK_DIR = Path("/content/chess-piece-detection")
else:
    WORK_DIR = Path("./notebooks_work")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print("IN_COLAB:", IN_COLAB)
print("WORK_DIR:", WORK_DIR.resolve())

## 1. Setup

Installs current wheels. After a successful Colab run, record the printed versions below if you need to pin.

In [ ]:
%pip install -q ultralytics roboflow coremltools pyyaml pillow matplotlib

In [ ]:
import importlib.metadata
import torch

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print(
        "WARNING: no GPU. Enable a Colab GPU before the train cell "
        "(Runtime → Change runtime type → T4)."
    )

for pkg in ("ultralytics", "roboflow", "coremltools"):
    print(f"{pkg} {importlib.metadata.version(pkg)}")

# Last known-good Colab combo (update after you smoke-install):
# ultralytics 8.3.x  |  roboflow 1.2.x  |  coremltools 8.x

## 2. Config

In [ ]:
from pathlib import Path

# Roboflow Universe: https://universe.roboflow.com/chess-ai-0uukd/chess-piece-detection-cpgnx/dataset/1
RF_WORKSPACE = "chess-ai-0uukd"
RF_PROJECT = "chess-piece-detection-cpgnx"
RF_VERSION = 1
RF_FORMAT = "yolov8"  # YOLO11 uses the same data.yaml + txt labels

MODEL_NAME = "yolo11n.pt"

IMGSZ = 640
BATCH = -1  # AutoBatch: use ~GPU memory
EPOCHS = 300
PATIENCE = 50  # early stop on mAP50-95
OPTIMIZER = "auto"
COS_LR = True
PRETRAINED = True
SAVE_PERIOD = 10
RUN_NAME = "yolo11n-chess-pieces"
PROJECT_DIR = WORK_DIR / "runs"

# Stronger augmentations for a full accuracy run (HSV left at Ultralytics defaults)
MOSAIC = 1.0
CLOSE_MOSAIC = 15
MIXUP = 0.15
COPY_PASTE = 0.1
DEGREES = 10.0
SCALE = 0.5
FLIPLR = 0.5  # no flipud — board photos are upright

if IN_COLAB:
    DATASET_DIR = Path("/content/datasets/chess-piece-detection")
else:
    DATASET_DIR = WORK_DIR / "datasets" / "chess-piece-detection"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

COREML_PATH = None  # set by the export cell

print("DATASET_DIR:", DATASET_DIR)
print("PROJECT_DIR:", PROJECT_DIR)
print("run:", RUN_NAME, "|", MODEL_NAME, "| epochs", EPOCHS, "| imgsz", IMGSZ)

## 3. Download dataset

Create a free key at [Roboflow](https://app.roboflow.com/) → Account → API. The notebook never prints or writes the key.

Public Universe downloads still require a key with Universe access.

In [ ]:
import getpass

from roboflow import Roboflow

api_key = getpass.getpass("ROBOFLOW_API_KEY: ")
if not api_key.strip():
    raise ValueError("A Roboflow API key is required to download the Universe dataset.")

try:
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    dataset = project.version(RF_VERSION).download(
        RF_FORMAT,
        location=str(DATASET_DIR),
        overwrite=False,
    )
except Exception as exc:
    raise RuntimeError(
        "Failed to download the Roboflow dataset. Check the API key and that "
        "this Universe project is public:\n"
        "https://universe.roboflow.com/chess-ai-0uukd/chess-piece-detection-cpgnx/dataset/1\n"
        f"workspace={RF_WORKSPACE}  project={RF_PROJECT}  version={RF_VERSION}\n"
        f"Original error: {exc}"
    ) from exc

del api_key
DATA_ROOT = Path(dataset.location)
print("downloaded to:", DATA_ROOT)

## 4. Inspect classes, splits, and samples

Roboflow's `data.yaml` often points at `../train/images` (one directory above the download). This cell locates the real `train/valid/test` folders, rewrites `data.yaml` to absolute paths so YOLO training works, and merges duplicate class names (`Black Bishop` vs `black-bishop`).

In [ ]:
from collections import Counter
import re
from pathlib import Path

import yaml

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CANONICAL_PIECES = [
    "black-bishop",
    "black-king",
    "black-knight",
    "black-pawn",
    "black-queen",
    "black-rook",
    "white-bishop",
    "white-king",
    "white-knight",
    "white-pawn",
    "white-queen",
    "white-rook",
]
PIECE_ALIASES = {re.sub(r"[^a-z0-9]+", "", name): name for name in CANONICAL_PIECES}


def find_data_yaml(root: Path) -> Path:
    direct = root / "data.yaml"
    if direct.exists():
        return direct
    matches = sorted(root.rglob("data.yaml"))
    if not matches:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return matches[0]


def class_names(raw) -> list[str]:
    if isinstance(raw, dict):
        return [str(raw[k]) for k in sorted(raw, key=lambda x: int(x))]
    return [str(n) for n in raw]


def list_images(images_dir: Path) -> list[Path]:
    if not images_dir.is_dir():
        return []
    return sorted(p for p in images_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)


def pair_if_populated(images: Path) -> tuple[Path, Path] | None:
    if images.name != "images" and (images / "images").is_dir():
        images = images / "images"
    if list_images(images):
        return images, images.parent / "labels"
    return None


def resolve_split(yaml_path: Path, raw, split: str, search_roots: list[Path]) -> tuple[Path, Path] | None:
    candidates: list[Path] = []
    if raw:
        path = Path(str(raw))
        if not path.is_absolute():
            path = (yaml_path.parent / path).resolve()
        candidates.append(path)
        # Roboflow writes ../train/images but files live next to data.yaml
        if len(path.parts) >= 2 and path.name == "images":
            candidates.append(yaml_path.parent / path.parent.name / "images")
    folders = {"train": ("train",), "val": ("valid", "val"), "test": ("test",)}[split]
    for root in search_roots:
        for folder in folders:
            candidates.append(root / folder / "images")
            candidates.append(root / folder)
    seen: set[Path] = set()
    for cand in candidates:
        cand = cand.resolve()
        if cand in seen:
            continue
        seen.add(cand)
        found = pair_if_populated(cand)
        if found:
            return found
    return None


def count_boxes(labels_dir: Path) -> Counter:
    counts: Counter = Counter()
    if not labels_dir.is_dir():
        return counts
    for txt in labels_dir.glob("*.txt"):
        for line in txt.read_text().splitlines():
            parts = line.split()
            if parts:
                counts[int(float(parts[0]))] += 1
    return counts


def canonicalize_name(name: str) -> str:
    key = re.sub(r"[^a-z0-9]+", "", name.lower())
    return PIECE_ALIASES.get(key, name)


def merge_duplicate_classes(names: list[str], splits: dict) -> list[str]:
    normalized = [canonicalize_name(n) for n in names]
    unique: list[str] = []
    for n in normalized:
        if n not in unique:
            unique.append(n)
    if set(unique) <= set(CANONICAL_PIECES):
        unique = [c for c in CANONICAL_PIECES if c in unique]
    old_to_new = {i: unique.index(normalized[i]) for i in range(len(names))}
    identity = old_to_new == {i: i for i in range(len(names))} and names == unique
    if identity:
        return names
    max_id = -1
    for _, labels_dir in splits.values():
        counts = count_boxes(labels_dir)
        if counts:
            max_id = max(max_id, max(counts))
    if max_id >= 0 and max_id < len(unique) and len(names) > len(unique):
        print(
            f"Labels already use ids 0–{max_id}; updating data.yaml names only "
            f"({len(names)} → {len(unique)})"
        )
        return unique
    print(f"Merging {len(names)} class names into {len(unique)}: {unique}")
    for _, labels_dir in splits.values():
        if not labels_dir.is_dir():
            continue
        for txt in labels_dir.glob("*.txt"):
            lines = []
            for line in txt.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                old = int(float(parts[0]))
                if old not in old_to_new:
                    continue
                parts[0] = str(old_to_new[old])
                lines.append(" ".join(parts))
            txt.write_text("\n".join(lines) + ("\n" if lines else ""))
    return unique


DATA_YAML = find_data_yaml(DATA_ROOT)
with DATA_YAML.open() as f:
    DATA = yaml.safe_load(f)

print("data.yaml:", DATA_YAML)
print("raw yaml paths:", {k: DATA.get(k) for k in ("train", "val", "valid", "test")})

search_roots = []
for root in (DATA_YAML.parent, DATA_ROOT, DATA_ROOT.parent):
    resolved = root.resolve()
    if resolved not in search_roots:
        search_roots.append(resolved)

SPLITS: dict[str, tuple[Path, Path]] = {}
for split, key in (("train", "train"), ("val", "val"), ("test", "test")):
    raw = DATA.get(key) or (DATA.get("valid") if split == "val" else None)
    found = resolve_split(DATA_YAML, raw, split, search_roots)
    if found:
        SPLITS[split] = found

if "train" not in SPLITS:
    print("Dataset tree (first 40 paths):")
    for i, path in enumerate(sorted(DATA_ROOT.rglob("*"))):
        if i >= 40:
            print("  ...")
            break
        print(" ", path.relative_to(DATA_ROOT), "dir" if path.is_dir() else "")
    raise FileNotFoundError(
        "No train images found. Roboflow data.yaml likely points at ../train/images "
        f"outside {DATA_YAML.parent}. Re-download or inspect the tree above."
    )

backup = DATA_YAML.with_suffix(".yaml.orig")
if not backup.exists():
    backup.write_text(DATA_YAML.read_text())

NAMES = merge_duplicate_classes(class_names(DATA.get("names", [])), SPLITS)
DATA["names"] = NAMES
DATA["nc"] = len(NAMES)
DATA["train"] = str(SPLITS["train"][0])
DATA["val"] = str(SPLITS.get("val", SPLITS["train"])[0])
if "test" in SPLITS:
    DATA["test"] = str(SPLITS["test"][0])
DATA.pop("valid", None)
DATA_YAML.write_text(yaml.safe_dump(DATA, sort_keys=False))

print("rewrote", DATA_YAML, "with absolute split paths")
print("nc:", DATA["nc"], "classes:", NAMES)
print()
for split, (images, labels) in SPLITS.items():
    n_img = len(list_images(images))
    box_counts = count_boxes(labels)
    print(f"{split}: {n_img} images, {sum(box_counts.values())} boxes  ({images})")
    for idx, name in enumerate(NAMES):
        print(f"    {idx:2d} {name:20s} {box_counts.get(idx, 0)}")
    print()

In [ ]:
%matplotlib inline

import random

import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image as PILImage

def draw_yolo(ax, image_path: Path, label_path: Path, names: list[str]) -> None:
    img = PILImage.open(image_path).convert("RGB")
    w, h = img.size
    ax.imshow(img)
    ax.set_axis_off()
    ax.set_title(image_path.name, fontsize=9)
    if not label_path.exists():
        return
    cmap = plt.cm.tab20.colors
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:5])
        x = (xc - bw / 2) * w
        y = (yc - bh / 2) * h
        color = cmap[cls % 20]
        ax.add_patch(
            patches.Rectangle((x, y), bw * w, bh * h, fill=False, edgecolor=color, linewidth=1.6)
        )
        label = names[cls] if 0 <= cls < len(names) else str(cls)
        ax.text(x, max(0, y - 2), label, color="white", fontsize=7, backgroundcolor=color)


train_images, train_labels = SPLITS.get("train", (None, None))
if train_images is None or not train_images.is_dir():
    raise FileNotFoundError("No train/images split found in the downloaded dataset.")

candidates = sorted(p for p in train_images.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
sample = random.sample(candidates, k=min(6, len(candidates)))

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, image_path in zip(axes.ravel(), sample):
    draw_yolo(ax, image_path, train_labels / f"{image_path.stem}.txt", NAMES)
for ax in axes.ravel()[len(sample) :]:
    ax.set_axis_off()
fig.suptitle("Train samples with YOLO boxes", fontsize=12)
fig.tight_layout()
plt.show()

## 5. Train YOLO11n

Full accuracy run: COCO-pretrained `yolo11n.pt`, 300 epochs, cosine LR, AutoBatch, early stopping (`patience=50`), checkpoint every 10 epochs.

If Colab disconnects, skip this cell and run **Resume** instead.

In [ ]:
import torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise SystemError(
        "No GPU detected. For a 300-epoch run use a Colab GPU "
        "(Runtime → Change runtime type → T4)."
    )

RUN_DIR = PROJECT_DIR / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    batch=BATCH,
    epochs=EPOCHS,
    patience=PATIENCE,
    optimizer=OPTIMIZER,
    cos_lr=COS_LR,
    pretrained=PRETRAINED,
    save_period=SAVE_PERIOD,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    mosaic=MOSAIC,
    close_mosaic=CLOSE_MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    degrees=DEGREES,
    scale=SCALE,
    fliplr=FLIPLR,
    device=0,
)

print("train dir:", RUN_DIR)
print("best.pt:", BEST_PT, "exists" if BEST_PT.exists() else "MISSING")
print("last.pt:", LAST_PT, "exists" if LAST_PT.exists() else "MISSING")

## 5b. Resume after a Colab disconnect

Leave `RESUME = False` for a normal Run-all. After a disconnect: re-run setup/config/dataset cells, set `RESUME = True` here, **skip Train**, and run this cell.

In [ ]:
from ultralytics import YOLO

RESUME = False  # True only after a Colab disconnect; skip the Train cell

LAST_PT = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"
if not RESUME:
    print("Skipping resume. Set RESUME = True after a disconnect, and skip Train.")
elif not LAST_PT.exists():
    raise FileNotFoundError(
        f"No checkpoint at {LAST_PT}. Mount Drive if you saved there, "
        "or re-run Train from scratch."
    )
else:
    resume_results = YOLO(str(LAST_PT)).train(resume=True)
    print("resumed from", LAST_PT)

## 6. Evaluate

mAP, precision/recall, confusion matrix, PR curve, and `results.png` from the training run.

In [ ]:
from pathlib import Path

from IPython.display import Image as IPyImage
from IPython.display import display
from ultralytics import YOLO

RUN_DIR = PROJECT_DIR / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
if not BEST_PT.exists():
    raise FileNotFoundError(f"Missing {BEST_PT}. Train or resume first.")

trained = YOLO(str(BEST_PT))
metrics = trained.val(data=str(DATA_YAML), plots=True, imgsz=IMGSZ)

box = metrics.box
print(f"mAP50     {box.map50:.4f}")
print(f"mAP50-95  {box.map:.4f}")
print(f"precision {box.mp:.4f}")
print(f"recall    {box.mr:.4f}")

plot_names = (
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
)
seen = set()
for plot in plot_names:
    for found in list(RUN_DIR.glob(plot)) + list(RUN_DIR.rglob(plot)):
        if found in seen:
            continue
        seen.add(found)
        print(found)
        display(IPyImage(filename=str(found), width=720))

## 6b. Sample predictions

In [ ]:
from pathlib import Path

from IPython.display import Image as IPyImage
from IPython.display import display
from ultralytics import YOLO

pred_split = SPLITS.get("val") or SPLITS.get("test") or SPLITS.get("train")
if pred_split is None:
    raise FileNotFoundError("No images split found for sample predictions.")
pred_images_dir, _ = pred_split
image_files = sorted(
    p for p in pred_images_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
if not image_files:
    raise FileNotFoundError(f"No images in {pred_images_dir}")

sample_paths = image_files[:8]
pred_out = PROJECT_DIR / RUN_NAME / "pred_samples"
pred_out.mkdir(parents=True, exist_ok=True)

trained = YOLO(str(PROJECT_DIR / RUN_NAME / "weights" / "best.pt"))
trained.predict(
    source=[str(p) for p in sample_paths],
    imgsz=IMGSZ,
    save=True,
    project=str(PROJECT_DIR / RUN_NAME),
    name="pred_samples",
    exist_ok=True,
)

saved = sorted(pred_out.glob("*"))
print("wrote", len(saved), "files to", pred_out)
for path in saved:
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        display(IPyImage(filename=str(path), width=640))

## 7. Export

`best.pt` / `last.pt` are already in the run `weights/` folder.

Primary Core ML export keeps full precision and embeds NMS. INT8 is optional and usually hurts accuracy.

If Core ML export fails on Colab (Linux + `coremltools`), download `best.pt` and export on a Mac later:

```python
from ultralytics import YOLO
YOLO("best.pt").export(format="coreml", nms=True, imgsz=640)
```

In [ ]:
from pathlib import Path

from ultralytics import YOLO

BEST_PT = PROJECT_DIR / RUN_NAME / "weights" / "best.pt"
LAST_PT = PROJECT_DIR / RUN_NAME / "weights" / "last.pt"
COREML_PATH = None

print("PyTorch weights")
print("  best.pt", BEST_PT, BEST_PT.exists())
print("  last.pt", LAST_PT, LAST_PT.exists())

try:
    COREML_PATH = YOLO(str(BEST_PT)).export(
        format="coreml",
        nms=True,
        imgsz=IMGSZ,
    )
    print("Core ML:", COREML_PATH)
except Exception as exc:
    print(
        "Core ML export failed on this platform (common on Colab Linux with NMS).\n"
        "Keep best.pt and re-export on macOS with:\n"
        '  YOLO("best.pt").export(format="coreml", nms=True, imgsz=640)\n'
        f"Error: {exc}"
    )

### Optional INT8 Core ML

Skip unless you need a smaller package. This is a second export and can reduce mAP.

In [ ]:
# Optional — leave commented unless you want a quantized copy.
# YOLO(str(BEST_PT)).export(format="coreml", nms=True, imgsz=IMGSZ, quantize=8)

## 8. Artifact checklist

In [ ]:
from pathlib import Path

RUN_DIR = PROJECT_DIR / RUN_NAME
weights = RUN_DIR / "weights"
print("Run directory:", RUN_DIR.resolve())
print()
print("Weights")
for name in ("best.pt", "last.pt"):
    path = weights / name
    print(f"  {path}  ({'ok' if path.exists() else 'missing'})")
print()
print("Core ML")
mlpackages = list(weights.glob("*.mlpackage")) + list(RUN_DIR.glob("*.mlpackage"))
if COREML_PATH:
    print("  export() returned:", COREML_PATH)
if mlpackages:
    for path in mlpackages:
        print(" ", path)
else:
    print("  (none — export skipped or failed; use best.pt on a Mac)")
print()
print("Plots")
for plot in ("results.png", "confusion_matrix.png", "PR_curve.png"):
    hits = list(RUN_DIR.glob(plot)) + list(RUN_DIR.rglob(plot))
    print(f"  {plot}: {hits[0] if hits else 'missing'}")
print()
print("Predictions:", RUN_DIR / "pred_samples")
print()
print(
    "Reminder: this is a full-image object detector, not a drop-in "
    "for Takes PieceClassifier.mlmodel (13-class square-crop classifier)."
)

## 9. Optional hyperparameter search

`model.tune()` is slow (hours on Colab). Leave this cell unrun unless you want to spend a full session on it. After a tune, retrain with the suggested hyps.

In [ ]:
# Optional / expensive. Uncomment to run.
#
# from ultralytics import YOLO
# YOLO(MODEL_NAME).tune(
#     data=str(DATA_YAML),
#     iterations=30,
#     epochs=50,
#     imgsz=IMGSZ,
#     batch=BATCH,
#     optimizer=OPTIMIZER,
#     project=str(PROJECT_DIR),
#     name=f"{RUN_NAME}-tune",
# )